# 🌌 Quantum-AI Sentinel — Comparative Analysis Notebook

This notebook benchmarks the **Hybrid Sentinel** (Classical + VQC) against the **Classical Baseline** across multiple datasets and metrics.

---
**Author:** Taha Erdem | **Project:** Quantum-AI Sentinel | **Institute:** PoliTO

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

from final_model import HybridSentinel, BaselineClassical, load_and_prepare_data, plot_loss_comparison
from data.data_utils import generate_spiral_dataset, generate_circles_dataset, train_test_split_manual, min_max_normalize

# Dark theme for plots
plt.style.use('dark_background')
HYBRID_COLOR    = '#58a6ff'
CLASSICAL_COLOR = '#f78166'
print('✅ Imports OK')

## 1. Dataset Overview

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.patch.set_facecolor('#0d1117')

datasets = {
    'Moons':   load_and_prepare_data('moons', 300),
    'Spirals': (lambda X, y: (X.T, None, y.reshape(1,-1), None))(*generate_spiral_dataset(300)),
    'Circles': (lambda X, y: (X.T, None, y.reshape(1,-1), None))(*generate_circles_dataset(300)),
}

titles = list(datasets.keys())
for ax, (name, data) in zip(axes, datasets.items()):
    X_tr = data[0]
    y_tr = data[2].flatten()
    ax.set_facecolor('#161b22')
    scatter = ax.scatter(X_tr[0], X_tr[1], c=y_tr, cmap='coolwarm', alpha=0.7, s=20)
    ax.set_title(name, color='white', fontsize=12, fontweight='bold')
    ax.tick_params(colors='white')
    for spine in ax.spines.values():
        spine.set_edgecolor('#30363d')

plt.suptitle('Benchmark Datasets', color='white', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('dataset_overview.png', dpi=120, bbox_inches='tight', facecolor='#0d1117')
plt.show()

## 2. Training: Hybrid Sentinel vs Classical Baseline (Moons)

In [ ]:
# Load moons dataset
X_train, X_test, y_train, y_test = load_and_prepare_data('moons', n_samples=300, noise=0.2)
input_dim = X_train.shape[0]
print(f'Train: {X_train.shape[1]} | Test: {X_test.shape[1]} | Features: {input_dim}')

In [ ]:
# Train Hybrid
print('🔵 Training Hybrid Sentinel …')
hybrid = HybridSentinel(input_dim=input_dim, n_qubits=4, q_layers=2, lr_classical=0.05, lr_quantum=0.01)
h_history = hybrid.train(X_train, y_train, epochs=60, verbose=True, log_every=10)

In [ ]:
# Train Classical
print('🔴 Training Classical Baseline …')
baseline = BaselineClassical(input_dim=input_dim, lr=0.05)
c_history = baseline.train(X_train, y_train, epochs=60, verbose=True, log_every=10)

In [ ]:
plot_loss_comparison(h_history, c_history, save_path='loss_comparison.png')

## 3. Decision Boundary Visualization

In [ ]:
def plot_decision_boundary(model, X, y, title, ax, use_predict=True):
    """Visualize the 2D decision boundary of a model."""
    x_min, x_max = X[0].min() - 0.5, X[0].max() + 0.5
    y_min, y_max = X[1].min() - 0.5, X[1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 60),
                          np.linspace(y_min, y_max, 60))
    grid = np.c_[xx.ravel(), yy.ravel()].T  # (2, N)
    Z = model.predict_proba(grid).reshape(xx.shape)

    ax.set_facecolor('#161b22')
    ax.contourf(xx, yy, Z, alpha=0.4, cmap='RdBu', levels=20)
    ax.scatter(X[0], X[1], c=y.flatten(), cmap='RdBu', edgecolors='white', linewidths=0.3, s=25)
    ax.set_title(title, color='white', fontsize=11, fontweight='bold')
    ax.tick_params(colors='white')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0d1117')

plot_decision_boundary(hybrid,   X_test, y_test, '🌌 Hybrid Sentinel',     axes[0])
plot_decision_boundary(baseline, X_test, y_test, '⚡ Classical Baseline',  axes[1])

plt.suptitle('Decision Boundaries — Test Set', color='white', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('decision_boundaries.png', dpi=120, bbox_inches='tight', facecolor='#0d1117')
plt.show()

## 4. Performance Summary

In [ ]:
h_pred   = hybrid.predict(X_test)
h_proba  = hybrid.predict_proba(X_test)
c_pred   = baseline.predict(X_test)
c_proba  = baseline.predict_proba(X_test)

results = {
    'Hybrid Sentinel': {
        'Accuracy': accuracy_score(y_test.flatten(), h_pred.flatten()),
        'ROC-AUC':  roc_auc_score(y_test.flatten(), h_proba.flatten()),
        'Final Loss': h_history[-1],
    },
    'Classical Baseline': {
        'Accuracy': accuracy_score(y_test.flatten(), c_pred.flatten()),
        'ROC-AUC':  roc_auc_score(y_test.flatten(), c_proba.flatten()),
        'Final Loss': c_history[-1],
    }
}

print('\n' + '='*50)
print(f'{"Metric":<20} {"Hybrid":>12} {"Classical":>12}')
print('='*50)
for metric in ['Accuracy', 'ROC-AUC', 'Final Loss']:
    h_val = results['Hybrid Sentinel'][metric]
    c_val = results['Classical Baseline'][metric]
    print(f'{metric:<20} {h_val:>12.4f} {c_val:>12.4f}')
print('='*50)

## 5. Quantum Circuit Diagram

In [ ]:
print('Variational Quantum Circuit (4 qubits, 2 layers):')
print('-' * 60)
hybrid.q_layer.circuit_diagram()

## 6. Quantum Parameter Heatmap

In [ ]:
params = hybrid.q_layer.params  # (n_layers, n_qubits, 2)

fig, axes = plt.subplots(1, params.shape[0], figsize=(10, 3))
fig.patch.set_facecolor('#0d1117')
if params.shape[0] == 1:
    axes = [axes]

for i, ax in enumerate(axes):
    ax.set_facecolor('#161b22')
    im = ax.imshow(params[i], cmap='plasma', aspect='auto')
    ax.set_title(f'Layer {i+1} Params', color='white', fontsize=10)
    ax.set_xlabel('Gate (RY, RZ)', color='white')
    ax.set_ylabel('Qubit', color='white')
    ax.tick_params(colors='white')
    plt.colorbar(im, ax=ax)

plt.suptitle('Trained Quantum Gate Parameters', color='white', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('quantum_params.png', dpi=120, bbox_inches='tight', facecolor='#0d1117')
plt.show()